In [ ]:
# Mount into drive

from google.colab import drive

drive.mount("/content/drive")

%cd '/content/drive/MyDrive/assignment3_NLP_spr26/'

!pip install -r requirements.txt




In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configuration
NUM_SAMPLES = 1000  # Sample dataset size
TEXT_EMBED_DIM = 768  # Typical BERT embedding size
IMAGE_EMBED_DIM = 2048  # Typical ResNet embedding size
NUM_GENRES = 20  # Number of movie genres in ML-20M
BATCH_SIZE = 32
LEARNING_RATE = 0.001

print("Configuration set up successfully!")

In [ ]:
# Generate random embeddings for demonstration
def generate_random_embeddings(num_samples, text_dim, image_dim, num_genres):
    """Generate random text, image embeddings and genre labels"""
    
    # Random text embeddings (simulating BERT/CLIP text embeddings)
    text_embeddings = torch.randn(num_samples, text_dim)
    
    # Random image embeddings (simulating ResNet/ViT image embeddings)
    image_embeddings = torch.randn(num_samples, image_dim)
    
    # Random multi-label genre assignments (simulating ML-20P dataset)
    genre_labels = torch.randint(0, 2, (num_samples, num_genres)).float()
    
    # Ensure each movie has at least one genre
    for i in range(num_samples):
        if genre_labels[i].sum() == 0:
            genre_labels[i, random.randint(0, num_genres-1)] = 1.0
    
    return text_embeddings, image_embeddings, genre_labels

# Generate sample data
text_embeds, image_embeds, genre_labels = generate_random_embeddings(
    NUM_SAMPLES, TEXT_EMBED_DIM, IMAGE_EMBED_DIM, NUM_GENRES
)

print(f"Generated {NUM_SAMPLES} samples:")
print(f"Text embeddings shape: {text_embeds.shape}")
print(f"Image embeddings shape: {image_embeds.shape}")
print(f"Genre labels shape: {genre_labels.shape}")
print(f"Average genres per movie: {genre_labels.sum(dim=1).mean():.2f}")

In [ ]:
class MovieDataset(Dataset):
    """Dataset class for multi-modal movie data"""
    
    def __init__(self, text_embeddings, image_embeddings, genre_labels):
        self.text_embeddings = text_embeddings
        self.image_embeddings = image_embeddings
        self.genre_labels = genre_labels
        
    def __len__(self):
        return len(self.text_embeddings)
    
    def __getitem__(self, idx):
        return {
            'text': self.text_embeddings[idx],
            'image': self.image_embeddings[idx],
            'genres': self.genre_labels[idx]
        }

def early_fusion(text_embeds, image_embeds):
    """
    Early Fusion: Concatenate text and image embeddings
    
    Args:
        text_embeds: Tensor of shape (batch_size, text_dim)
        image_embeds: Tensor of shape (batch_size, image_dim)
    
    Returns:
        fused_embeds: Tensor of shape (batch_size, text_dim + image_dim)
    """
    # Normalize embeddings to prevent scale imbalance
    text_embeds = nn.functional.normalize(text_embeds, p=2, dim=1)
    image_embeds = nn.functional.normalize(image_embeds, p=2, dim=1)
    
    # Concatenate along feature dimension
    fused_embeds = torch.cat([text_embeds, image_embeds], dim=1)
    
    return fused_embeds

# Test early fusion
sample_text = text_embeds[:4]
sample_image = image_embeds[:4]
fused = early_fusion(sample_text, sample_image)

print(f"Early Fusion Test:")
print(f"Text shape: {sample_text.shape}")
print(f"Image shape: {sample_image.shape}")
print(f"Fused shape: {fused.shape}")
print(f"Fused dimension: {TEXT_EMBED_DIM + IMAGE_EMBED_DIM}")

In [ ]:
class EarlyFusionMovieClassifier(nn.Module):
    """Neural network for multi-label movie genre classification using early fusion"""
    
    def __init__(self, text_dim, image_dim, num_genres, hidden_dims=[1024, 512, 256]):
        super(EarlyFusionMovieClassifier, self).__init__()
        
        # Input dimension after fusion
        fused_dim = text_dim + image_dim
        
        # Build the classifier layers
        layers = []
        prev_dim = fused_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.BatchNorm1d(hidden_dim)
            ])
            prev_dim = hidden_dim
        
        # Output layer for multi-label classification
        layers.append(nn.Linear(prev_dim, num_genres))
        
        self.classifier = nn.Sequential(*layers)
        
    def forward(self, text_embeds, image_embeds):
        # Apply early fusion
        fused_embeds = early_fusion(text_embeds, image_embeds)
        
        # Pass through classifier
        logits = self.classifier(fused_embeds)
        
        return logits

# Initialize the model
model = EarlyFusionMovieClassifier(
    text_dim=TEXT_EMBED_DIM,
    image_dim=IMAGE_EMBED_DIM,
    num_genres=NUM_GENRES
)

print(f"Model initialized with {sum(p.numel() for p in model.parameters())} parameters")
print(model)

In [ ]:
# Split data into train and test sets
train_size = int(0.8 * NUM_SAMPLES)
test_size = NUM_SAMPLES - train_size

train_dataset = MovieDataset(
    text_embeds[:train_size],
    image_embeds[:train_size],
    genre_labels[:train_size]
)

test_dataset = MovieDataset(
    text_embeds[train_size:],
    image_embeds[train_size:],
    genre_labels[train_size:]
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")
print(f"Number of batches: {len(train_loader)}")

In [ ]:
# Training setup
criterion = nn.BCEWithLogitsLoss()  # Multi-label classification
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
num_epochs = 10

# Training loop
def train_model(model, train_loader, criterion, optimizer, num_epochs):
    model.train()
    train_losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        
        for batch in train_loader:
            text = batch['text']
            image = batch['image']
            genres = batch['genres']
            
            # Forward pass
            outputs = model(text, image)
            loss = criterion(outputs, genres)
            
            # Backward pass and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1
        
        avg_loss = epoch_loss / num_batches
        train_losses.append(avg_loss)
        
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')
    
    return train_losses

# Train the model
print("Starting training...")
train_losses = train_model(model, train_loader, criterion, optimizer, num_epochs)
print("Training completed!")

In [ ]:
# Evaluation
def evaluate_model(model, test_loader, criterion):
    model.eval()
    total_loss = 0.0
    correct_predictions = 0
    total_predictions = 0
    
    with torch.no_grad():
        for batch in test_loader:
            text = batch['text']
            image = batch['image']
            genres = batch['genres']
            
            outputs = model(text, image)
            loss = criterion(outputs, genres)
            total_loss += loss.item()
            
            # Calculate accuracy (multi-label)
            predictions = torch.sigmoid(outputs) > 0.5
            correct_predictions += (predictions == genres.bool()).sum().item()
            total_predictions += genres.numel()
    
    avg_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions
    
    return avg_loss, accuracy

# Evaluate the model
test_loss, test_accuracy = evaluate_model(model, test_loader, criterion)

print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

# Show a sample prediction
sample_batch = next(iter(test_loader))
with torch.no_grad():
    text_sample = sample_batch['text'][:3]
    image_sample = sample_batch['image'][:3]
    genre_sample = sample_batch['genres'][:3]
    
    predictions = torch.sigmoid(model(text_sample, image_sample))
    
print("\nSample Predictions:")
for i in range(3):
    pred_genres = (predictions[i] > 0.5).nonzero().squeeze().tolist()
    true_genres = genre_sample[i].nonzero().squeeze().tolist()
    print(f"Movie {i+1}: Predicted genres: {pred_genres}, True genres: {true_genres}")